# 01 — Data Exploration

**Goal:** Understand the financial time-series data before building anomaly detection models.

We'll examine:
1. Data shape, types, and missing values
2. Price and volume distributions
3. Return distributions (are they normal? heavy-tailed?)
4. Correlation structure between features
5. Known anomaly events visualized in context

**Data Source:** Yahoo Finance via `yfinance` (OHLCV daily data)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

from src.data.fetcher import DataFetcher
from src.utils.helpers import load_config

plt.style.use('seaborn-v0_8-darkgrid')
config = load_config('../config/config.yaml')

## 1. Load Data

In [ ]:
fetcher = DataFetcher(config)

# Load all tickers
tickers = config['data']['tickers']
data = {}
for t in tickers:
    data[t] = fetcher.fetch(ticker=t)

# Primary ticker for detailed analysis
ticker = 'SPY'
df = data[ticker]
print(f"\n--- {ticker} ---")
print(f"Shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isna().sum()}")
print(f"\nBasic statistics:")
df.describe()

## 2. Price and Volume Over Time

Visualize the raw OHLCV data to get a sense of trends, volatility regimes, and where known events fall.

In [ ]:
# Price + Volume chart with known events highlighted
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                    row_heights=[0.7, 0.3], subplot_titles=['Close Price', 'Volume'])

fig.add_trace(go.Scatter(x=df.index, y=df['Close'], name='Close',
                         line=dict(color='#2196F3', width=1)), row=1, col=1)

fig.add_trace(go.Bar(x=df.index, y=df['Volume'], name='Volume',
                     marker_color='rgba(38,166,154,0.5)'), row=2, col=1)

# Shade known events
events = config['evaluation']['known_events']
colors = ['rgba(255,0,0,0.15)', 'rgba(255,165,0,0.15)', 'rgba(148,0,211,0.15)']
for i, event in enumerate(events):
    for row in [1, 2]:
        fig.add_vrect(x0=event['start'], x1=event['end'],
                      fillcolor=colors[i % len(colors)], layer='below',
                      line_width=0, row=row, col=1,
                      annotation_text=event['name'] if row == 1 else None,
                      annotation_position='top left')

fig.update_layout(height=600, template='plotly_dark', title=f'{ticker} — Price & Volume (5Y)',
                  showlegend=False)
fig.show()

## 3. Return Distribution Analysis

Financial returns are famously **not** normally distributed — they have heavier tails (more extreme events than a Gaussian predicts). This is exactly why anomaly detection matters.

We'll test for normality and measure tail heaviness (kurtosis).

In [ ]:
returns = df['Close'].pct_change().dropna()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Histogram vs Normal
axes[0].hist(returns, bins=80, density=True, alpha=0.7, color='#2196F3', label='Actual')
x = np.linspace(returns.min(), returns.max(), 200)
axes[0].plot(x, stats.norm.pdf(x, returns.mean(), returns.std()),
             'r-', linewidth=2, label='Normal fit')
axes[0].set_title(f'{ticker} Daily Returns vs Normal Distribution')
axes[0].set_xlabel('Daily Return')
axes[0].legend()

# Q-Q plot
stats.probplot(returns, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot (Normal)')
axes[1].get_lines()[0].set_color('#2196F3')

# Log returns over time (shows volatility clustering)
axes[2].plot(returns.index, returns, alpha=0.7, linewidth=0.5, color='#2196F3')
axes[2].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[2].axhline(y=returns.std()*3, color='red', linestyle='--', alpha=0.5, label='+3 sigma')
axes[2].axhline(y=-returns.std()*3, color='red', linestyle='--', alpha=0.5, label='-3 sigma')
axes[2].set_title('Daily Returns Over Time')
axes[2].set_xlabel('Date')
axes[2].legend()

plt.tight_layout()
plt.show()

# Normality statistics
print("=== Return Distribution Statistics ===")
print(f"Mean:       {returns.mean():.6f}")
print(f"Std Dev:    {returns.std():.6f}")
print(f"Skewness:   {returns.skew():.4f}  (normal = 0)")
print(f"Kurtosis:   {returns.kurtosis():.4f}  (normal = 0, heavy tails > 0)")
print(f"Min return: {returns.min():.4f}  ({returns.min()/returns.std():.1f} sigma)")
print(f"Max return: {returns.max():.4f}  ({returns.max()/returns.std():.1f} sigma)")

# Shapiro-Wilk test (subsample for large N)
sample = returns.sample(min(500, len(returns)), random_state=42)
stat_val, p_val = stats.shapiro(sample)
print(f"\nShapiro-Wilk test: statistic={stat_val:.4f}, p-value={p_val:.2e}")
print(f"Normal? {'Yes' if p_val > 0.05 else 'No'} (p {'>' if p_val > 0.05 else '<'} 0.05)")

# Count extreme events
n_beyond_3sigma = (returns.abs() > 3 * returns.std()).sum()
expected_3sigma = len(returns) * 0.0027  # 0.27% for normal
print(f"\nDays beyond 3-sigma: {n_beyond_3sigma} (expected if normal: {expected_3sigma:.1f})")
print(f"Ratio: {n_beyond_3sigma/max(expected_3sigma,0.1):.1f}x more extreme events than normal predicts")

## 4. Cross-Ticker Comparison

Compare return distributions and volatility across tickers to see if anomaly patterns are asset-specific or market-wide.

In [ ]:
# Compare return distributions across tickers
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

summary = []
for i, (t, t_df) in enumerate(data.items()):
    ret = t_df['Close'].pct_change().dropna()
    axes[i].hist(ret, bins=60, density=True, alpha=0.7, color='#2196F3')
    axes[i].set_title(f'{t} Daily Returns')
    axes[i].set_xlabel('Return')
    axes[i].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    
    summary.append({
        'Ticker': t,
        'Observations': len(ret),
        'Mean Return': f"{ret.mean():.5f}",
        'Volatility': f"{ret.std():.5f}",
        'Skewness': f"{ret.skew():.3f}",
        'Kurtosis': f"{ret.kurtosis():.3f}",
        'Max Drawdown': f"{ret.min():.4f}",
        'Days > 3%': int((ret.abs() > 0.03).sum()),
    })

plt.tight_layout()
plt.show()

pd.DataFrame(summary).set_index('Ticker')

## 5. Correlation Structure

Examine how price features relate to each other. Strong correlations between features can cause some models to double-count information.

In [ ]:
# Correlation heatmap of raw OHLCV + returns
df_corr = df.copy()
df_corr['returns'] = df_corr['Close'].pct_change()
df_corr['log_returns'] = np.log(df_corr['Close'] / df_corr['Close'].shift(1))
df_corr['volatility_20d'] = df_corr['returns'].rolling(20).std()
df_corr = df_corr.dropna()

corr = df_corr[['Open', 'High', 'Low', 'Close', 'Volume', 'returns', 'volatility_20d']].corr()

fig = go.Figure(data=go.Heatmap(
    z=corr.values, x=corr.columns, y=corr.columns,
    text=np.round(corr.values, 2), texttemplate='%{text}',
    colorscale='RdBu_r', zmid=0
))
fig.update_layout(title=f'{ticker} — Feature Correlation Matrix',
                  template='plotly_dark', height=500, width=600)
fig.show()

## 6. Volatility Clustering

Financial markets exhibit **volatility clustering** — periods of high volatility tend to be followed by more high volatility. This is a well-documented stylized fact (Mandelbrot, 1963) and is the reason EWMA-based detectors work.

In [ ]:
# Rolling volatility + autocorrelation of squared returns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rolling 20-day volatility
vol = returns.rolling(20).std() * np.sqrt(252)  # annualized
axes[0].plot(vol.index, vol, color='#FF9800', linewidth=0.8)
axes[0].set_title(f'{ticker} — Rolling 20-Day Annualized Volatility')
axes[0].set_ylabel('Volatility')
axes[0].set_xlabel('Date')

# Autocorrelation of |returns| (evidence of volatility clustering)
from pandas.plotting import autocorrelation_plot
abs_returns = returns.abs()
lags = range(1, 41)
acf = [abs_returns.autocorr(lag=l) for l in lags]
axes[1].bar(lags, acf, color='#2196F3', alpha=0.7)
axes[1].axhline(y=1.96/np.sqrt(len(abs_returns)), color='red', linestyle='--', alpha=0.5)
axes[1].axhline(y=-1.96/np.sqrt(len(abs_returns)), color='red', linestyle='--', alpha=0.5)
axes[1].set_title('Autocorrelation of |Returns| (Volatility Clustering)')
axes[1].set_xlabel('Lag (days)')
axes[1].set_ylabel('Autocorrelation')

plt.tight_layout()
plt.show()

print("Significant positive autocorrelation in |returns| confirms volatility clustering.")
print("This means anomalies tend to cluster in time — an isolated spike is less common than a burst.")

## Key Takeaways

1. **Returns are not normal** — heavy tails with excess kurtosis. There are significantly more extreme events than a Gaussian distribution predicts. This justifies using anomaly detection rather than simple z-score thresholds.

2. **Volatility clusters** — high-volatility periods persist. Anomaly detectors need to account for changing volatility regimes (EWMA does this, static z-score does not).

3. **Volume spikes correlate with price moves** — but not always. Some high-volume days have normal returns, and some extreme price moves happen on normal volume. This is why we use multiple features.

4. **BTC-USD is fundamentally different** — much higher volatility, more extreme returns. A model trained on SPY may not transfer well to crypto.

5. **Known events are visible** — COVID crash (Feb-Mar 2020), GME squeeze (Jan 2021), SVB collapse (Mar 2023) all show clear signatures in both price and volume.

**Next:** Feature engineering to extract signals from this raw data.